In [3]:
import os
import csv
import random
from typing import Any, List

import numpy as np
import pandas as pd
import torch
from keras.callbacks import EarlyStopping
from numpy import floating
from sklearn.metrics import average_precision_score
from sklearn.model_selection import KFold, train_test_split

from MLPModel import MLPModel

settings = {
    "model": {
        "mlp": {
            "lr": [0.005],
            "n_folds": 10,
            "epochs": 100,
            "k": 50,
            "iter": 10,
            "dropout_rates": [0.2],
            "neurons_per_layer": [[512, 1024, 2048]],
            "model_names": ["pair_mapped_only_256"],
        }
    }
}

BASE_PATH = os.getcwd()
LOG_PATH = os.path.join(BASE_PATH, "logs")
os.makedirs(LOG_PATH, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def log(columns: List[Any], values: List[Any], filepath: str) -> None:
    file_exists = os.path.isfile(filepath)
    with open(filepath, mode="a", newline="") as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(columns)
        writer.writerow(values)


def shuffle_df(df: pd.DataFrame, frac: float = 1.0, random_state: int = 42) -> pd.DataFrame:
    return df.sample(frac=frac, random_state=random_state).reset_index(drop=True)


def average_precision_at_k_multi_label(y_true: pd.DataFrame, y_pred: np.ndarray, k: int = 50) -> floating:
    ap_at_k_list = []
    for i in range(y_true.shape[1]):
        sorted_indices = np.argsort(y_pred[:, i])[::-1][:k]
        sorted_true = y_true.iloc[sorted_indices, i]
        ap_at_k_label = average_precision_score(sorted_true, y_pred[sorted_indices, i])
        ap_at_k_list.append(ap_at_k_label)
    return np.mean(ap_at_k_list)


set_seed(12)

pair_path = os.path.join(BASE_PATH, "pair_embeddings_mapped_only.pt")
if not os.path.exists(pair_path):
    raise FileNotFoundError(f"Missing pair embeddings file: {pair_path}")

pair_obj = torch.load(pair_path, map_location="cpu", weights_only=False)

# Accept common saved formats: dict, tuple/list, or explicit tensor keys.
if isinstance(pair_obj, dict):
    x_key = next((k for k in ["x", "X", "features", "embeddings"] if k in pair_obj), None)
    y_key = next((k for k in ["y", "Y", "labels", "targets"] if k in pair_obj), None)

    if x_key is not None and y_key is not None:
        X_tensor = torch.as_tensor(pair_obj[x_key]).float()
        y_tensor = torch.as_tensor(pair_obj[y_key]).float()
    else:
        raise ValueError(
            "Dict format detected, but couldn't find X/Y keys. "
            "Expected one of x/X/features/embeddings and y/Y/labels/targets."
        )
elif isinstance(pair_obj, (tuple, list)) and len(pair_obj) == 2:
    X_tensor = torch.as_tensor(pair_obj[0]).float()
    y_tensor = torch.as_tensor(pair_obj[1]).float()
else:
    raise ValueError(
        "Unsupported pair_embeddings_mapped_only.pt format. "
        "Use dict with X/Y keys or tuple/list as (X, y)."
    )

if X_tensor.dim() != 2 or y_tensor.dim() != 2:
    raise ValueError(f"Expected 2D tensors, got X:{tuple(X_tensor.shape)}, y:{tuple(y_tensor.shape)}")
if X_tensor.size(0) != y_tensor.size(0):
    raise ValueError(f"Row mismatch: X has {X_tensor.size(0)} rows, y has {y_tensor.size(0)} rows")

print("Loaded pair-mapped-only embeddings")
print("X shape:", tuple(X_tensor.shape))
print("y shape:", tuple(y_tensor.shape))
print("Embedding dim (expected 256):", X_tensor.shape[1])

X_df = pd.DataFrame(X_tensor.numpy())
y_df = pd.DataFrame(y_tensor.numpy())

for model_name in settings["model"]["mlp"]["model_names"]:
    for neurons_per_layer in settings["model"]["mlp"]["neurons_per_layer"]:
        for lr_rate in settings["model"]["mlp"]["lr"]:
            for dr_rate in settings["model"]["mlp"]["dropout_rates"]:
                print("\n")
                print(f"=================       {model_name}      =================")
                print("\n")

                X = shuffle_df(X_df)
                y = shuffle_df(y_df)
                X_train_val, X_test, y_train_val, y_test = train_test_split(
                    X, y, test_size=0.1, random_state=1
                )

                print("\nSizes:")
                print(f"\tOriginal X size: {X.shape}")
                print(f"\ty size: {y.shape}")
                print(f"\tX cross-validation size: {X_train_val.shape}")
                print(f"\tX_test size: {X_test.shape}")

                n_inputs, n_outputs = X_train_val.shape[1], y.shape[1]
                print("\nI/O Model Size:")
                print(f"\t Input Size: {n_inputs}")
                print(f"\t Output Size: {n_outputs}")

                for iteration in range(settings["model"]["mlp"]["iter"]):
                    print("Iteration:", iteration + 1)
                    fold_counter = 0
                    early_stopping = EarlyStopping(monitor="val_AUC", mode="max", verbose=1)
                    kf = KFold(
                        n_splits=settings["model"]["mlp"]["n_folds"],
                        shuffle=True,
                        random_state=42,
                    )

                    builder = MLPModel(lr_rate)
                    builder.create_architecture(n_inputs, n_outputs, neurons_per_layer, dr_rate)
                    model = builder.compile()

                    print("\n")
                    print("================================================")
                    print("                  Training Phase                ")
                    print("================================================")
                    print("\n")

                    for train_index, validation_index in kf.split(X_train_val, y_train_val):
                        print(f"\n\t Fold: {fold_counter + 1} \n")
                        fold_counter = fold_counter + 1

                        X_train = X_train_val.iloc[train_index]
                        X_validation = X_train_val.iloc[validation_index]
                        y_train = y_train_val.iloc[train_index]
                        y_validation = y_train_val.iloc[validation_index]

                        model.fit(
                            X_train,
                            y_train,
                            validation_data=(X_validation, y_validation),
                            verbose=1,
                            epochs=settings["model"]["mlp"]["epochs"],
                            callbacks=[early_stopping],
                            batch_size=128,
                        )

                    print("\n")
                    print("================================================")
                    print("                    Test Phase                  ")
                    print("================================================")
                    print("\n")

                    test_metrics = model.evaluate(X_test, y_test, verbose=1)
                    y_pred = model.predict(X_test, verbose=1)
                    k = settings["model"]["mlp"]["k"]
                    test_ap_at_k = average_precision_at_k_multi_label(y_test, y_pred, k=k)
                    print(test_ap_at_k)

                    log(
                        columns=["Model", "Loss", "AUC", "AUPRC", f"AP@{k}", "Comment"],
                        values=[
                            model_name,
                            test_metrics[0],
                            test_metrics[1],
                            test_metrics[2],
                            test_ap_at_k,
                            "pair_mapped_only_256",
                        ],
                        filepath=os.path.join(LOG_PATH, "mlp_pair_mapped_only_256.csv"),
                    )


Loaded pair-mapped-only embeddings
X shape: (48006, 256)
y shape: (48006, 963)
Embedding dim (expected 256): 256


=================       pair_mapped_only_256      =================



Sizes:
	Original X size: (48006, 256)
	y size: (48006, 963)
	X cross-validation size: (43205, 256)
	X_test size: (4801, 256)

I/O Model Size:
	 Input Size: 256
	 Output Size: 963
Iteration: 1


                  Training Phase                



	 Fold: 1 



c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - AUC: 0.6533 - AUPRC: 0.1187 - loss: 0.0821 - val_AUC: 0.6912 - val_AUPRC: 0.1397 - val_loss: 0.1790
Epoch 2/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - AUC: 0.7534 - AUPRC: 0.1865 - loss: 0.0648 - val_AUC: 0.7651 - val_AUPRC: 0.2029 - val_loss: 0.0709
Epoch 3/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - AUC: 0.7844 - AUPRC: 0.2108 - loss: 0.0629 - val_AUC: 0.7964 - val_AUPRC: 0.2316 - val_loss: 0.0632
Epoch 4/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - AUC: 0.8048 - AUPRC: 0.2305 - loss: 0.0615 - val_AUC: 0.8117 - val_AUPRC: 0.2505 - val_loss: 0.0616
Epoch 5/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - AUC: 0.8186 - AUPRC: 0.2459 - loss: 0.0605 - val_AUC: 0.8206 - val_AUPRC: 0.2596 - val_loss: 0.0608
Epoch 6/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - AUC: 0.8300 - AUPRC: 0.2605 - loss: 0.0595 - val_AUC: 0.8300 - val_AUPRC: 0.2777 - val_loss: 0.0600
Epoch 7/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - AUC: 0.

c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - AUC: 0.6532 - AUPRC: 0.1188 - loss: 0.0826 - val_AUC: 0.6925 - val_AUPRC: 0.1400 - val_loss: 0.2438
Epoch 2/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.7527 - AUPRC: 0.1852 - loss: 0.0649 - val_AUC: 0.7704 - val_AUPRC: 0.2020 - val_loss: 0.0760
Epoch 3/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - AUC: 0.7845 - AUPRC: 0.2109 - loss: 0.0629 - val_AUC: 0.7973 - val_AUPRC: 0.2375 - val_loss: 0.0625
Epoch 4/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.8036 - AUPRC: 0.2298 - loss: 0.0616 - val_AUC: 0.8117 - val_AUPRC: 0.2485 - val_loss: 0.0617
Epoch 5/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.8183 - AUPRC: 0.2468 - loss: 0.0605 - val_AUC: 0.8236 - val_AUPRC: 0.2676 - val_loss: 0.0607
Epoch 6/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - AUC: 0.8296 - AUPRC: 0.2618 - loss: 0.0595 - val_AUC: 0.8287 - val_AUPRC: 0.2756 - val_loss: 0.0601
Epoch 7/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - AUC: 0.

...

Epoch 1/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.6504 - AUPRC: 0.1181 - loss: 0.0815 - val_AUC: 0.6847 - val_AUPRC: 0.1354 - val_loss: 0.1867
Epoch 2/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.7531 - AUPRC: 0.1871 - loss: 0.0648 - val_AUC: 0.7713 - val_AUPRC: 0.2114 - val_loss: 0.0696
Epoch 3/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - AUC: 0.7853 - AUPRC: 0.2135 - loss: 0.0628 - val_AUC: 0.7977 - val_AUPRC: 0.2362 - val_loss: 0.0631
Epoch 4/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.8052 - AUPRC: 0.2316 - loss: 0.0615 - val_AUC: 0.8120 - val_AUPRC: 0.2518 - val_loss: 0.0619
Epoch 5/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.8204 - AUPRC: 0.2494 - loss: 0.0603 - val_AUC: 0.8247 - val_AUPRC: 0.2672 - val_loss: 0.0604
Epoch 6/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.8307 - AUPRC: 0.2632 - loss: 0.0594 - val_AUC: 0.8325 - val_AUPRC: 0.2782 - val_loss: 0.0597
Epoch 7/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - AUC: 0.

c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - AUC: 0.6510 - AUPRC: 0.1188 - loss: 0.0820 - val_AUC: 0.6738 - val_AUPRC: 0.1263 - val_loss: 0.1962
Epoch 2/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.7533 - AUPRC: 0.1869 - loss: 0.0649 - val_AUC: 0.7693 - val_AUPRC: 0.2028 - val_loss: 0.0777
Epoch 3/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.7846 - AUPRC: 0.2111 - loss: 0.0629 - val_AUC: 0.7969 - val_AUPRC: 0.2358 - val_loss: 0.0631
Epoch 4/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - AUC: 0.8045 - AUPRC: 0.2293 - loss: 0.0616 - val_AUC: 0.8118 - val_AUPRC: 0.2511 - val_loss: 0.0613
Epoch 5/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - AUC: 0.8196 - AUPRC: 0.2476 - loss: 0.0604 - val_AUC: 0.8238 - val_AUPRC: 0.2681 - val_loss: 0.0604
Epoch 6/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - AUC: 0.8313 - AUPRC: 0.2638 - loss: 0.0594 - val_AUC: 0.8315 - val_AUPRC: 0.2805 - val_loss: 0.0598
Epoch 7/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.

c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\activations\leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 12s 28ms/step - AUC: 0.6517 - AUPRC: 0.1189 - loss: 0.0808 - val_AUC: 0.6888 - val_AUPRC: 0.1391 - val_loss: 0.1520
Epoch 2/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - AUC: 0.7536 - AUPRC: 0.1859 - loss: 0.0649 - val_AUC: 0.7721 - val_AUPRC: 0.2075 - val_loss: 0.0744
Epoch 3/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - AUC: 0.7861 - AUPRC: 0.2130 - loss: 0.0628 - val_AUC: 0.7982 - val_AUPRC: 0.2390 - val_loss: 0.0634
Epoch 4/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - AUC: 0.8052 - AUPRC: 0.2318 - loss: 0.0615 - val_AUC: 0.8115 - val_AUPRC: 0.2540 - val_loss: 0.0615
Epoch 5/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - AUC: 0.8206 - AUPRC: 0.2492 - loss: 0.0603 - val_AUC: 0.8226 - val_AUPRC: 0.2666 - val_loss: 0.0610
Epoch 6/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step - AUC: 0.8318 - AUPRC: 0.2646 - loss: 0.0593 - val_AUC: 0.8326 - val_AUPRC: 0.2820 - val_loss: 0.0598
Epoch 7/100
304/304 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - AUC: 0.